[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/mp-2/blob/main/notebooks/01_graphical_method.ipynb)

# Графический метод

Ноутбук строит допустимую область, проверяет вершины и показывает, где достигаются минимум и максимум.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

c = np.array([-3.0, 6.0])
constraints = [
    (np.array([5.0, -2.0]), "<=", 4.0, "5*x1 - 2*x2 <= 4"),
    (np.array([1.0, -2.0]), ">=", -4.0, "x1 - 2*x2 >= -4"),
    (np.array([1.0, 1.0]), ">=", 4.0, "x1 + x2 >= 4"),
]

def objective(point):
    return float(c @ point)

def feasible(point, tol=1e-9):
    if point[0] < -tol or point[1] < -tol:
        return False
    for a, sense, b, _ in constraints:
        left = float(a @ point)
        if sense == "<=" and left > b + tol:
            return False
        if sense == ">=" and left < b - tol:
            return False
    return True

def active(point, tol=1e-7):
    names = []
    if abs(point[0]) <= tol:
        names.append("x1 = 0")
    if abs(point[1]) <= tol:
        names.append("x2 = 0")
    for a, _, b, label in constraints:
        if abs(float(a @ point) - b) <= tol:
            names.append(label)
    return "; ".join(names)

from itertools import combinations

boundaries = constraints + [
    (np.array([1.0, 0.0]), ">=", 0.0, "x1 >= 0"),
    (np.array([0.0, 1.0]), ">=", 0.0, "x2 >= 0"),
]

points = []
for first, second in combinations(boundaries, 2):
    a1, _, b1, _ = first
    a2, _, b2, _ = second
    matrix = np.vstack([a1, a2])
    if abs(np.linalg.det(matrix)) < 1e-9:
        continue
    point = np.linalg.solve(matrix, np.array([b1, b2]))
    if feasible(point):
        if not any(np.linalg.norm(point - old, ord=np.inf) < 1e-7 for old in points):
            points.append(point)

rows = []
for index, point in enumerate(sorted(points, key=lambda p: (p[0], p[1])), start=1):
    rows.append({
        "point": f"A{index}",
        "x1": point[0],
        "x2": point[1],
        "Z": objective(point),
        "active constraints": active(point),
    })

vertices = pd.DataFrame(rows)
vertices

In [ ]:
print('minimum')
display(vertices.loc[[vertices['Z'].idxmin()]])
print('maximum')
display(vertices[abs(vertices['Z'] - vertices['Z'].max()) < 1e-7])

In [ ]:
polygon = vertices[["x1", "x2"]].to_numpy()
center = polygon.mean(axis=0)
order = np.argsort(np.arctan2(polygon[:, 1] - center[1], polygon[:, 0] - center[0]))
polygon = polygon[order]

x_values = np.linspace(0, 3, 300)
fig, ax = plt.subplots(figsize=(7, 5), dpi=130)
ax.fill(polygon[:, 0], polygon[:, 1], alpha=0.35, color="#8ecae6", label="feasible region")
ax.plot(np.r_[polygon[:, 0], polygon[0, 0]], np.r_[polygon[:, 1], polygon[0, 1]], color="#126782")

for a, _, b, label in constraints:
    if abs(a[1]) > 1e-12:
        ax.plot(x_values, (b - a[0] * x_values) / a[1], label=label)

max_value = vertices["Z"].max()
min_value = vertices["Z"].min()
max_rows = vertices[abs(vertices["Z"] - max_value) < 1e-7]
min_row = vertices.loc[vertices["Z"].idxmin()]
ax.scatter(max_rows["x1"], max_rows["x2"], marker="s", color="#fb8500", label="max Z")
ax.scatter([min_row["x1"]], [min_row["x2"]], marker="o", color="#111111", label="min Z")

ax.set_xlim(0, 2.4)
ax.set_ylim(2.0, 3.2)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.show()